In [334]:
#EXemplo pequeno
import os
import glob
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from docplex.cp.model import CpoModel
import math

'''
def ler_instancia_pequena():

    m=3 #qtd de máquinas
    n=5 #qtd de operações
    
    A=[[0,1],[1,2],[3,4]] #conj. que indica as precedências entre operações de um job. ex. [0,1] -> op 0 vem antes da op 1
    F=[[0,1],[1],[2],[1],[0,2]] # indica as máquinas que podem processar uma operação. ex. [1,2] -> op. 0 pode ser processada pela máquina 1 e 2
    P=[[7,6,0],[0,4,0],[0,0,7],[0,6,0],[4,0,6]] # tempo de processamento de cada operação e máquina. ex. [7,6,0] -> tempo para processar op. 0 nas máquinas 1, 2 e 3 
    Of=[2,4]
    return m, n, A, F, P, D, Of
'''

def ler_instancia_pequena():

    m=4 #4#qtd de máquinas
    n=12 #qtd de operações 
    
    A=[[0,1],[0,2],[1,3],[2,3],[4,6],[5,6],[6,7],[8,9],[9,11],[10,11]] #conj. que indica as precedências entre operações de um job. ex. [0,1] -> op 0 vem antes da op 1
    F=[[0],[0,1],[2],[1],[2,3],[3],[0,2],[0,3],[1],[0,2],[3],[1,2]] # indica as máquinas que podem processar uma operação. ex. [[1,2],...] -> op. 0 pode ser processada pela máquina 1 e 2
    P=[[5,0,0,0],[7,7,0,0],[0,0,9,0],[0,11,0,0],[0,0,15,15],[0,0,0,10],[20,0,20,0],[5,0,0,5],[0,8,0,0],[12,0,12,0],[0,0,0,20],[0,5,5,0]] # tempo de processamento de cada operação e máquina. ex. [7,6,0] -> tempo para processar op. 0 nas máquinas 1, 2 e 3 
    D=[0,0,0,35,0,0,0,35,0,35,0,35]
    Of=[3,7,11]

    return m, n, A, F, P, D, Of


In [28]:
#leitura dos dados de entrada - ajuste o caminho do arquivo
#Instâncias non-linear routes

import os
import glob
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from docplex.cp.model import CpoModel
import math

def ler_instancia_fjsp(caminho_instancia,g):
    with open(caminho_instancia, 'r') as f:
        lines = [lines.strip() for lines in f.readlines() if lines.strip()]

    # Primeira linha: número de operações, relações de precedência e máquinas
    first_line = list(map(int, lines[0].split()))
    n_operations = first_line[0]
    n_precedence = first_line[1]
    n_machines = first_line[2]
    FFp = lines[1 + n_precedence + n_operations]
    
    A = []  # Precedências (i1, i2)
    F = [[] for _ in range(n_operations)]  # Máquinas que podem processar cada operação
    P = [[] for _ in range(n_operations)]  # Tempos de processamento para cada máquina
    D = [0] * n_operations  # Datas de entrega (inicialmente todas 0)
    Of = []  # Operações finais (serão identificadas depois)

#A
    for i in range(1, n_precedence + 1):
        i1, i2 = map(int, lines[i].split())
        A.append([i1, i2])

#F e P
    op_lines = lines[n_precedence + 1 : n_precedence + 1 + n_operations]
    for i in range(n_operations):
        parts = list(map(int, op_lines[i].split()))
        num_machines_for_op = parts[0]
        machines_and_times = parts[1:]
        
        P[i] = [0] * n_machines
        
        for j in range(num_machines_for_op):
            machine = machines_and_times[2 * j]
            time = machines_and_times[2 * j + 1]
            F[i].append(machine)
            P[i][machine]= time  
#D
    all_operations = set(range(n_operations))
    operations_with_successors = set(i1 for (i1, i2) in A)
    Of = list(all_operations - operations_with_successors)
    Of.sort()

    jobs = []
    current_job = []
    all_ops = sorted(set(op for pair in A for op in pair))  # Todas operações com precedência
    print(f'all_ops {all_ops}')
    
    for op in range(n_operations):
        current_job.append(op)
        if op in Of:  # Se é uma operação final, fecha o job
            jobs.append(current_job)
            current_job = []
    
    valores = FFp.strip().split()
    valores_numericos = [v for v in valores if v != '.' and v != '']
    datas = list(map(float, valores_numericos))
    print(f'datas {datas}')

    datas = [math.floor(x * g) for x in datas]
    
    D=[0]* len(range(n_operations))
    for v, t in zip(Of, datas):
        D[v] = t 


    
    print(f"m={n_machines}")
    print(f"n={n_operations}")
    print(f"A={A}")
    print(f"F={F}")
    print(f"P={P}")
    print(f"D={D}")
    print(f"Of={Of}")
    
    return n_machines, n_operations, A, F, P, D, Of



#m, n, A, F, P, D, Of = ler_instancia_fjsp(caminho_instancia)

In [32]:
#Definição do modelo
#Lunardi et al 2020 reduzido com FO de minimização de atraso

def CreateModel(m, n, A, F, P, D, Of):
    model = CpoModel()
    H = 0
    for i in range(n):
        for j in range(len(F[i])):
            H += P[i][F[i][j]]
                
    #Variáveis
    op = [model.interval_var(name="O" + str(i)) for i in range(n)] #variável de intervalo para operações
    opm = [[model.interval_var(optional=True, size=P[i][F[i][j]], name="O" + str(i) + "M" + str(F[i][j])) for j in range(len(F[i]))]  for i in range(n)] #variável de intervalo maquina operação 
    seq = [model.sequence_var([opm[i][j] for i in range(n) for j in range(len(F[i])) if F[i][j] == k], name="Seq" + str(k)) for k in range(m)] #variável de sequência máquinas

    #Função objetivo
    model.add(model.minimize(model.sum(model.max(0, model.end_of(op[i])-D[i]) for i in Of))) #minimização atraso
    #model.add(model.minimize(model.max(model.end_of(op[i]) for i in range(n)))) #minimização makespan

    #Restrições
    for s in A:
        model.add(model.end_before_start(op[s[0]], op[s[1]])) #restrição 34
        print(f"op {s[0]} termina antes de op {s[1]} começar")

    for i in range(n):
        model.add(model.alternative(op[i], opm[i])) #restrição 35
        #print(f"Restrição alternativa: operação {i} com variáveis {[v.get_name() for v in opm[i]]}")
    for k in range(m):
        model.add(model.no_overlap(seq[k]))
        #print(f"Restrição no_overlap: sequência {seq[k]}")

    return model, op, opm

def solve(m, n, A, F, P, D, Of, nome_saida, DIR_RESULTADOS):
    model, op, opm = CreateModel(m, n, A, F, P, D, Of)
    #sol = model.solve(TimeLimit=3600)
    nome_arquivo_modelo = os.path.join(DIR_RESULTADOS, f"{nome_saida}.cpo")
    print("processando:",DIR_RESULTADOS,nome_saida)
    model.export_model(nome_arquivo_modelo)
        
    if os.path.exists(nome_arquivo_modelo):
        print(f"    Arquivo exportado com sucesso!")
        tamanho = os.path.getsize(nome_arquivo_modelo)
        print(f"    Tamanho do arquivo: {tamanho} bytes")
    else:
        print(f"    Falha ao exportar arquivo!")

    #print(f" - Objetivo (Objective Value): {sol.get_objective_value()}\n")   #<<<<<<<<<<<<<<<<<<<
                
    #return sol
    return model

In [40]:
# Configurar diretórios
DIR_INSTANCIAS = "C:\\Users\\Hanna Pamplona\\Doutorado\\Instâncias\\non-linear routes\\2. com FIFO\\DA_FFp"
DIR_RESULTADOS = "C:\\Users\\Hanna Pamplona\\Doutorado\\Instâncias\\non-linear routes\\3. LPs\\sem block\\DA.cpo"

valores_g = [0.25, 0.5, 0.75]
tipos = ["tight", "moderate", "loose"]

for arquivo in glob.glob(os.path.join(DIR_INSTANCIAS, "*.txt")):
    
    for g, tipo in zip(valores_g, tipos):
        print(f"\n=== Processando g={g}, tipo={tipo} ===")
        nome_base = os.path.basename(arquivo).replace("_FFp.txt", "")
        nome_saida = f"CP_{nome_base}_{tipo}_sem_block"  
        print(f"Processando: {nome_saida}")  
       
        try:
            m, n, A, F, P, D, Of = ler_instancia_fjsp(arquivo, g)
            #resultado = solve(m, m, A, F, P, D, Of, nome_saida, DIR_RESULTADOS)
            modelo = solve(m, n, A, F, P, D, Of, nome_saida, DIR_RESULTADOS)

        
        except Exception as e:
            print(f"Erro em {nome_saida}: {str(e)}")
        
        # Salvar resultados
        #caminho_txt = os.path.join(DIR_RESULTADOS, f"{nome_saida}_resultado.txt")
        #salvar_resultados_txt(resultado, caminho_txt)
        
        # Salvar Gantt se solução encontrada



=== Processando g=0.25, tipo=tight ===
Processando: CP_DAFJS01_tight_sem_block
all_ops [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]
datas [277.0216049382716, 182.3733108108108, 289.78985507246375, 373.57963446475196]
m=5
n=26
A=[[0, 1], [1, 2], [2, 3], [3, 8], [4, 5], [5, 6], [6, 7], [7, 8], [9, 10], [10, 13], [9, 11], [11, 13], [9, 12], [12, 13], [14, 15], [15, 18], [14, 16], [16, 18], [14, 17], [17, 18], [19, 20], [20, 21], [21, 25], [22, 23], [23, 24], [24, 25]]
F=[[0, 2, 1, 3], [3, 0, 2], [4, 0, 3], [1, 3], [2, 3, 1], [2, 1, 0, 3], [4, 1, 2, 3], [4, 1], [0, 4, 3], [3, 0, 1, 4], [2, 3, 0, 1], [1, 2, 4], [4, 1], [4, 2, 0], [0, 1, 3, 4], [2, 1], [4, 3, 2], [4, 3, 1, 2], [0, 3, 2], [0, 1, 4, 3], [4, 3], [4, 3, 1, 0], [2, 0, 1, 4], [1, 2, 0, 4], [4, 3], [2, 4]]
P=[[84, 87, 84, 91, 0], [77, 0, 70, 66, 0], [35, 0, 0, 23, 13], [0, 2, 0, 4, 0], [0, 99, 98, 98, 0], [51, 69, 24, 30, 0], [0, 40, 83, 51, 35], [0, 1, 0, 0, 1], [79, 0, 0, 86, 90]

In [34]:
## pequenas instâncias
DIR_INSTANCIAS_PEQUENAS = "C:\\Users\\Hanna Pamplona\\Doutorado\\Instâncias"
nome_saida="Problema pequeno sem blocking"
caminho_txt = os.path.join(DIR_INSTANCIAS_PEQUENAS, f"{nome_saida}_resultado.txt")
caminho_png = os.path.join(DIR_INSTANCIAS_PEQUENAS, f"{nome_saida}_gantt.png")

try:
    m, n, A, F, P, D, Of = ler_instancia_pequena()
    resultado = solve(m, n, A, F, P, D, Of, "Problema pequeno", DIR_INSTANCIAS_PEQUENAS)
    salvar_resultados_txt(resultado, caminho_txt, m, n)   
    salvar_gantt(resultado, Of, caminho_png, 70)
except Exception as e:
    print(f"Erro em {nome_saida}: {str(e)}")


Erro em Problema pequeno sem blocking: name 'ler_instancia_pequena' is not defined


In [31]:
def salvar_resultados_txt(resultado, caminho_saida, m, n):
    with open(caminho_saida, 'w') as f:
        if resultado is None:
            f.write("Nenhuma solução encontrada.\n")
            return
        
        f.write("Solução encontrada:\n")
        f.write(f"Objective Value:{resultado.get_objective_value()}\n")
        f.write(f"Lower Bound (LB):{resultado.get_objective_bound()}\n")

        gap=resultado.get_objective_gap()
        if gap is not None:
            f.write(f"GAP(%):{gap*100:.2f}\n")
        
        f.write(f"Solve Time (s):{resultado.get_solve_time():.2f}\n")        

        for i in range(n):
            try:
                valor = resultado.get_value(f"O{i}")
                f.write(f"  O{i}: {valor}\n")
            except:
                f.write(f"  O{i}: não encontrada\n")

        for i in range(n):
            for j in range(len(F[i])):
                nome_var = f"O{i}M{F[i][j]}"
                try:
                    valor = resultado.get_value(nome_var)
                    f.write(f"  {nome_var}: {valor}\n")
                except:
                    pass  
        for k in range(m):
            try:
                seq_value = resultado.get_value(f"Seq{k}")
                f.write(f"  Seq{k}: [")
                nomes = [v.get_name() for v in seq_value]
                f.write(", ".join(nomes))
                f.write("]\n")
            except:
                f.write(f"  Seq{k}: não disponível\n")

def salvar_gantt(resultado, Of, caminho_imagem, divisor):
    dados = []
    for i in range(n):
        for j in range(len(F[i])):
            nome_var = f"O{i}M{F[i][j]}"
            intervalo = resultado.get_value(nome_var)
        
        # se não for vazio (None) e tiver start/end
            if intervalo is not None and hasattr(intervalo, 'start') and intervalo.start is not None:
                dados.append((i, F[i][j], intervalo.start, intervalo.end))
        
        
        
    jobs = []
    inicio = 0
    for fim in Of:
        job_ops = list(range(inicio, fim + 1))
        jobs.append(job_ops)
        inicio = fim + 1
    
    # Mapear operações para jobs
    op_to_job = {}
    for job_id, job_ops in enumerate(jobs):
        for op in job_ops:
            op_to_job[op] = job_id
    
    # Criar gráfico
    fig, ax = plt.subplots(figsize=(20, 5))
    cores = plt.cm.tab20.colors
    maquinas = sorted(set(m for _, m, _, _ in dados))
    
    for op, m, inicio, fim in dados:
        job_id = op_to_job[op]
        ax.add_patch(patches.Rectangle(
            (inicio/divisor, m - 0.4),
            (fim - inicio)/divisor,
            0.8,
            facecolor=cores[job_id % len(cores)],
            edgecolor='black',
            label=f'Job {job_id}' if op == jobs[job_id][0] else ""
        ))
        ax.text((inicio + fim)/2/divisor, m, f"{op}", ha='center', va='center', fontsize=8, color='black')
    
    # Configurações do gráfico
    ax.set_yticks(maquinas)
    ax.set_yticklabels([f"Máquina {m}" for m in maquinas])
    ax.set_xlabel("Tempo")
    ax.set_ylabel("Máquinas")
    ax.set_title("Gráfico de Gantt - CP Lunardi^TT_block")
    ax.grid(True, axis='x', linestyle='--', alpha=0.5)
    ax.set_ylim(min(maquinas) - 1, max(maquinas) + 1)
    
    # Legenda
    handles, labels = ax.get_legend_handles_labels()
    unique_labels = dict(zip(labels, handles))
    ax.legend(unique_labels.values(), unique_labels.keys(), title="Jobs", bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.savefig(caminho_imagem, bbox_inches='tight')
    plt.close()